# Explore taxonomy RDF — local SPARQL queries

Uses `rdflib` to load the generated TTL bundles locally and run SPARQL queries interactively.  
No server required — edit queries in-place and re-run cells.

Once a query looks good, copy it to **[SPARQLQueries](https://github.com/pathway-lod/SPARQLQueries)** and add it to **[Snorql-UI](https://github.com/pathway-lod/Snorql-UI)**.

**Kernel:** select `plantmetwiki-rdf` (register once with `python -m ipykernel install --user --name plantmetwiki-rdf`).

---

## ⚠️ Limitations — read before writing queries

`rdflib` is a pure-Python in-memory SPARQL engine with **no query optimizer**:

| ✅ Works well | ❌ Avoid — will hang for minutes/hours |
|---|---|
| Simple `SELECT` with 1–2 triple patterns on one graph | **Merging graphs** with `g_a + g_b` — creates massive combined graph |
| `COUNT` / `GROUP BY` on one variable | `FILTER(STRSTARTS(?a, STR(?b)))` over large sets — O(n²) cartesian product |
| Queries on taxonomy bundle (~4 MB, fast) | Joining two triple patterns with no shared variable |
| Properties bundle queries (load separately, ~160 MB) | Queries on merged `g_both` — slow even if individual graphs are fast |

**URI structure** (pathway ≠ DataNode prefix — do NOT use STRSTARTS to link them):
```
Pathway:  http://rdf-plantmetwiki.bioinformatics.nl/pathways/PC1_r<version>
DataNode: http://rdf-plantmetwiki.bioinformatics.nl/Pathway/PC1_r<version>/DataNode/<id>
```

**For cross-layer queries** (pathway ↔ DataNode ↔ species), use the **Virtuoso** endpoint. See section 5.

In [ ]:
from rdflib import Graph, Namespace, URIRef
import pandas as pd
from pathlib import Path

PREFIXES = """
PREFIX wp:      <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi:    <http://purl.obolibrary.org/obo/NCBITaxon_>
PREFIX pmw:     <http://rdf-plantmetwiki.bioinformatics.nl/vocab/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
"""

def sparql(g: Graph, query: str) -> pd.DataFrame:
    """Run a SPARQL SELECT and return a DataFrame."""
    results = g.query(PREFIXES + query)
    return pd.DataFrame(results, columns=[str(v) for v in results.vars])

print("Ready.")

## Load bundles — keep them SEPARATE

**Do not merge** `g_tax + g_prop` — the properties bundle has hundreds of thousands of blank nodes and the combined graph is extremely slow to query in rdflib. Query each graph independently.

In [ ]:
VERSION = "plantcyc17.0.0-gpml2021"
bundles = Path("../output/bundles")

# ── Taxonomy extra (~4 MB, fast) ──────────────────────────────────────────────
print("Loading taxonomy extra...")
g_tax = Graph()
g_tax.parse(str(bundles / f"all_gpml_taxonomy_extra-{VERSION}.ttl"), format="turtle")
print(f"  {len(g_tax):,} triples  (all wp:organism statements)")

# ── Properties extra (~160 MB, ~1 min) ───────────────────────────────────────
print("Loading properties extra (this takes ~1 min)...")
g_prop = Graph()
g_prop.parse(str(bundles / f"all_gpml_properties_extra-{VERSION}.ttl"), format="turtle")
print(f"  {len(g_prop):,} triples  (pmw:gpmlProperty blank nodes + pmw:plantcycId)")

---
## 1. Pathway IRIs and PlantCyc IDs

Each pathway has a stable IRI (`/pathways/PC{n}_r{version}`) and a PlantCyc ID stored as `pmw:plantcycId`.

In [ ]:
# Pathway IRI + PlantCyc ID + identifiers.org source — sample 10
sparql(g_prop, """
SELECT ?pathway_iri ?plantcyc_id ?source
WHERE {
    ?pathway_iri pmw:plantcycId  ?plantcyc_id .
    ?pathway_iri dcterms:source  ?source .
    FILTER(CONTAINS(STR(?pathway_iri), "/pathways/"))
}
LIMIT 10
""")

In [ ]:
# Total pathways with a PlantCyc ID — expected: 1162
sparql(g_prop, """
SELECT (COUNT(DISTINCT ?iri) AS ?pathways_with_plantcyc_id)
WHERE {
    ?iri pmw:plantcycId ?id .
    FILTER(CONTAINS(STR(?iri), "/pathways/"))
}
""")

---
## 2. Verify taxonomy and properties use the same IRIs

Both bundles use the same pathway IRI as subject — they are implicitly linked by shared URI.  
We verify this in Python (fast set comparison) rather than a slow SPARQL join.

In [ ]:
from rdflib import URIRef

WP_ORGANISM  = URIRef("http://vocabularies.wikipathways.org/wp#organism")
PMW_ID       = URIRef("http://rdf-plantmetwiki.bioinformatics.nl/vocab/plantcycId")
VIRIDIPLANTAE = URIRef("http://purl.obolibrary.org/obo/NCBITaxon_33090")

# Pathway IRIs in taxonomy bundle (those with Viridiplantae + /pathways/ in URI)
tax_pathways  = {str(s) for s, p, o in g_tax
                 if p == WP_ORGANISM and o == VIRIDIPLANTAE
                 and "/pathways/" in str(s)}

# Pathway IRIs in properties bundle
prop_pathways = {str(s) for s, p, o in g_prop
                 if p == PMW_ID
                 and "/pathways/" in str(s)}

print(f"Pathway IRIs in taxonomy extra:   {len(tax_pathways):,}")
print(f"Pathway IRIs in properties extra: {len(prop_pathways):,}")
print(f"IRIs in both (shared):            {len(tax_pathways & prop_pathways):,}")
print(f"Only in taxonomy:                 {len(tax_pathways - prop_pathways):,}")
print(f"Only in properties:               {len(prop_pathways - tax_pathways):,}")

---
## 3. Explore GPML properties

PlantCyc key-value properties are stored as blank nodes: `?subject pmw:gpmlProperty [ pmw:key "..."; pmw:value "..." ]`

In [ ]:
# All unique property keys and how often they appear — top 20
sparql(g_prop, """
SELECT ?key (COUNT(?key) AS ?count)
WHERE {
    ?subject pmw:gpmlProperty ?bn .
    ?bn pmw:key ?key .
}
GROUP BY ?key
ORDER BY DESC(?count)
LIMIT 20
""")

In [ ]:
# Original per-pathway species list (preserved from PlantCyc before Viridiplantae fix)
sparql(g_prop, """
SELECT ?pathway_iri ?original_species
WHERE {
    ?pathway_iri pmw:gpmlProperty ?bn .
    ?bn pmw:key   "Organism" ;
        pmw:value ?original_species .
    FILTER(CONTAINS(STR(?pathway_iri), "/pathways/"))
}
LIMIT 10
""")

In [ ]:
# DataNode UniqueID values (PlantCyc gene/protein/compound IDs)
sparql(g_prop, """
SELECT ?datanode ?plantcyc_id
WHERE {
    ?datanode pmw:gpmlProperty ?bn .
    ?bn pmw:key   "UniqueID" ;
        pmw:value ?plantcyc_id .
    FILTER(CONTAINS(STR(?datanode), "/DataNode/"))
}
LIMIT 10
""")

---
## 4. Taxonomy queries

All run on the small `g_tax` graph only.

In [ ]:
# Count resources with Viridiplantae — expected 2478 (1162 pathways + 1316 reactions)
sparql(g_tax, """
SELECT (COUNT(DISTINCT ?resource) AS ?resources_with_viridiplantae)
WHERE { ?resource wp:organism ncbi:33090 . }
""")

In [ ]:
# Species distribution across DataNode URIs — top 20
sparql(g_tax, """
SELECT ?species (COUNT(DISTINCT ?node) AS ?node_count)
WHERE {
    ?node wp:organism ?species .
    FILTER(?species != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
}
GROUP BY ?species
ORDER BY DESC(?node_count)
LIMIT 20
""")

In [ ]:
# Sample DataNode + biological entity URIs for Arabidopsis thaliana (ncbi:3702)
sparql(g_tax, """
SELECT ?uri
WHERE { ?uri wp:organism ncbi:3702 . }
LIMIT 10
""")

In [ ]:
# UniProt URIs with species annotation
sparql(g_tax, """
SELECT ?entity ?species
WHERE {
    ?entity wp:organism ?species .
    FILTER(STRSTARTS(STR(?entity), "https://identifiers.org/uniprot/"))
}
LIMIT 10
""")

---
## 5. Cross-layer queries → Virtuoso only

These need `wp:isPartOf` from the core bundle. Run on the **Virtuoso endpoint** using named graphs.

```sparql
# Pathways containing at least one Arabidopsis thaliana DataNode
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi: <http://purl.obolibrary.org/obo/NCBITaxon_>

SELECT ?pathway (COUNT(DISTINCT ?node) AS ?arabidopsis_nodes)
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways> {
      ?node wp:isPartOf ?pathway .
  }
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-taxonomy-extra> {
      ?node wp:organism ncbi:3702 .
  }
}
GROUP BY ?pathway
ORDER BY DESC(?arabidopsis_nodes)
LIMIT 20
```

```sparql
# Pathway IRI + PlantCyc ID + Viridiplantae annotation
PREFIX wp:      <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi:    <http://purl.obolibrary.org/obo/NCBITaxon_>
PREFIX pmw:     <http://rdf-plantmetwiki.bioinformatics.nl/vocab/>

SELECT ?pathway_iri ?plantcyc_id ?organism
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-properties-extra> {
      ?pathway_iri pmw:plantcycId ?plantcyc_id .
  }
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-taxonomy-extra> {
      ?pathway_iri wp:organism ?organism .
  }
}
LIMIT 20
```

---
## 6. Sandbox

Use `g_tax` or `g_prop` — **not both together**. When a query is validated, copy it to **[SPARQLQueries](https://github.com/pathway-lod/SPARQLQueries)**.

In [ ]:
# Replace g_tax with g_prop if querying properties
sparql(g_tax, """
SELECT *
WHERE {
    # ← write your query here
}
LIMIT 20
""")